# Summary_Day14_offline.ipynb  
## 사전학습 모델 활용 2 · 전이학습 전략 비교 · 인터넷 불가 버전

이 파일은 **인터넷이 안 되는 환경**에서 14강의 전이학습 전략 흐름을 연습하기 위한 버전이다.

원본 강의는 CIFAR-10 다운로드와 ImageNet pretrained ResNet18 가중치를 사용한다.  
인터넷이 없으면 둘 다 실패할 수 있다.

그래서 오프라인 버전은 다음 방식으로 구성한다.

```text
데이터 다운로드 없음 → FakeData 사용
사전학습 가중치 다운로드 없음 → ResNet18(weights=None) 사용
구조 연습은 동일 → freeze, partial, full, differential LR, augmentation
```

> 주의:  
> `weights=None`은 진짜 사전학습 효과를 보여주지는 않는다.  
> 이 파일은 인터넷 없이 코드 구조를 연습하기 위한 대체 버전이다.

## 1. 전체 실습 목적

오프라인 실습의 목적은 다음이다.

1. FakeData로 CIFAR-10과 비슷한 10 class 이미지 데이터를 만든다.
2. ResNet18 구조를 `weights=None`으로 불러온다.
3. Full Freeze, Partial, Full 전략의 코드 차이를 확인한다.
4. `requires_grad`와 학습 가능한 parameter 수를 확인한다.
5. parameter group으로 차등 학습률 구조를 만든다.
6. Tensor 기반으로 weak/medium/strong augmentation을 흉내 낸다.
7. 작은 데이터셋 전략 선택 기준을 정리한다.

## 2. 라이브러리 준비

FakeData는 torchvision 안에 들어 있는 가짜 이미지 데이터셋이다.  
다운로드 없이 바로 만들 수 있다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

num_classes = len(class_names)

print("device:", device)
print("class 수:", num_classes)

## 3. FakeData 데이터셋 만들기

### 함수 사용법

```python
datasets.FakeData(size=500, image_size=(3, 224, 224), num_classes=10)
```

- `size`: 샘플 개수다.
- `image_size`: 이미지 shape이다.
- `num_classes`: class 개수다.
- 인터넷 다운로드가 필요 없다.

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

transform_basic = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

train_dataset = datasets.FakeData(
    size=500,
    image_size=(3, 224, 224),
    num_classes=num_classes,
    transform=transform_basic
)

test_dataset = datasets.FakeData(
    size=200,
    image_size=(3, 224, 224),
    num_classes=num_classes,
    transform=transform_basic
)

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

images, labels = next(iter(train_loader))

print("images:", images.shape)
print("labels:", labels.shape)

## 4. 이미지 시각화

FakeData는 랜덤 이미지라 의미 있는 사물은 아니지만, 이미지 분류 코드 흐름은 동일하다.

In [ ]:
def denormalize(img):
    img = img.numpy().transpose((1, 2, 0))
    mean = np.array(imagenet_mean)
    std = np.array(imagenet_std)
    img = std * img + mean
    img = np.clip(img, 0, 1)
    return img


def show_images(images, labels, num_images=5):
    fig, axes = plt.subplots(1, num_images, figsize=(15, 3))

    for i in range(num_images):
        axes[i].imshow(denormalize(images[i]))
        axes[i].set_title(class_names[labels[i].item()])
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()

show_images(images, labels, num_images=5)

## 5. ResNet18 구조 불러오기

인터넷이 없으므로 사전학습 가중치를 받지 않는다.

### 함수 사용법

```python
models.resnet18(weights=None)
```

- 모델 구조만 만든다.
- ImageNet pretrained 효과는 없다.
- freeze/partial/full 구조 연습에는 사용할 수 있다.

In [ ]:
def load_resnet18_offline():
    return models.resnet18(weights=None)

model = load_resnet18_offline()

print(model.fc)
print("fc in_features:", model.fc.in_features)

## 6. build_model 함수

온라인 버전과 같은 전략을 오프라인에서도 연습한다.

```text
freeze → fc만 학습
partial → layer4 + fc 학습
full → 전체 학습
```

In [ ]:
def count_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


def build_model(strategy):
    model = load_resnet18_offline()

    if strategy == "freeze":
        for param in model.parameters():
            param.requires_grad = False

        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif strategy == "partial":
        for param in model.parameters():
            param.requires_grad = False

        for param in model.layer4.parameters():
            param.requires_grad = True

        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif strategy == "full":
        for param in model.parameters():
            param.requires_grad = True

        model.fc = nn.Linear(model.fc.in_features, num_classes)

    else:
        raise ValueError("strategy는 freeze, partial, full 중 하나여야 한다.")

    return model.to(device)


for strategy in ["freeze", "partial", "full"]:
    model = build_model(strategy)
    total, trainable = count_trainable_params(model)
    print(f"{strategy:8s} | total={total:,} | trainable={trainable:,}")

## 7. 학습/평가 함수

FakeData는 랜덤 데이터라 정확도 자체는 의미가 크지 않다.  
여기서는 학습 루프 구조를 확인하는 것이 목적이다.

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

        _, predicted = outputs.max(1)

        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, 100.0 * correct / total


def evaluate_model(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)

            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / total, 100.0 * correct / total

## 8. Optimizer 만들기

전략별로 Optimizer 대상이 다르다.

In [ ]:
def make_optimizer(model, strategy, learning_rate=0.001):
    if strategy == "freeze":
        optimizer = optim.Adam(model.fc.parameters(), lr=learning_rate)

    elif strategy == "partial":
        optimizer = optim.Adam(
            [
                {"params": model.layer4.parameters()},
                {"params": model.fc.parameters()}
            ],
            lr=learning_rate
        )

    elif strategy == "full":
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    else:
        raise ValueError("strategy는 freeze, partial, full 중 하나여야 한다.")

    return optimizer

## 9. 전략 실행 함수와 짧은 학습

오프라인에서는 실행 시간을 줄이기 위해 각 전략을 1 epoch만 돌린다.

In [ ]:
def run_strategy(strategy, num_epochs=1):
    model = build_model(strategy)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(model, strategy)

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_acc": []
    }

    start_time = time.time()

    print(f"\n[{strategy}] 학습 시작")

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        _, test_acc = evaluate_model(model, test_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)

        print(
            f"Epoch [{epoch + 1}/{num_epochs}] - "
            f"Train Loss: {train_loss:.4f}, "
            f"Train Acc: {train_acc:.2f}%, "
            f"Test Acc: {test_acc:.2f}%"
        )

    elapsed_time = time.time() - start_time

    return history, elapsed_time


num_epochs = 1

history_freeze, elapsed_freeze = run_strategy("freeze", num_epochs)
history_partial, elapsed_partial = run_strategy("partial", num_epochs)
history_full, elapsed_full = run_strategy("full", num_epochs)

## 10. 결과 비교 표

FakeData에서는 성능 자체보다 표를 만드는 흐름을 본다.

In [ ]:
results = [
    [
        "Full Freeze",
        history_freeze["test_acc"][-1],
        elapsed_freeze,
        history_freeze["train_acc"][-1] - history_freeze["test_acc"][-1]
    ],
    [
        "Partial Fine-tune",
        history_partial["test_acc"][-1],
        elapsed_partial,
        history_partial["train_acc"][-1] - history_partial["test_acc"][-1]
    ],
    [
        "Full Fine-tune",
        history_full["test_acc"][-1],
        elapsed_full,
        history_full["train_acc"][-1] - history_full["test_acc"][-1]
    ]
]

print(f'{"Strategy":<20} {"Test Acc (%)":<15} {"Time (sec)":<15} {"Overfit Gap (%)":<15}')
print("-" * 80)

for result in results:
    print(f"{result[0]:<20} {result[1]:<15.2f} {result[2]:<15.2f} {result[3]:<15.2f}")

## 11. 차등 학습률 구조 확인

오프라인에서도 parameter group 구조는 그대로 연습할 수 있다.

In [ ]:
model_diff_lr = build_model("partial")

optimizer_diff = optim.Adam(
    [
        {"params": model_diff_lr.layer4.parameters(), "lr": 0.001},
        {"params": model_diff_lr.fc.parameters(), "lr": 0.01}
    ]
)

for i, group in enumerate(optimizer_diff.param_groups):
    print(f"group {i} learning rate:", group["lr"])

## 12. Tensor 기반 증강 함수

FakeData Tensor에 대해 간단한 augmentation을 직접 적용한다.

```text
weak → 좌우 반전
medium → 좌우 반전 + 약한 noise
strong → 좌우 반전 + noise + random erase
```

In [ ]:
def augment_batch(images, strength="weak"):
    images = images.clone()

    if strength in ["weak", "medium", "strong"]:
        if np.random.rand() < 0.5:
            images = torch.flip(images, dims=[3])

    if strength in ["medium", "strong"]:
        noise = 0.05 * torch.randn_like(images)
        images = images + noise

    if strength == "strong":
        batch_size = images.size(0)

        for i in range(batch_size):
            h = np.random.randint(20, 60)
            w = np.random.randint(20, 60)
            top = np.random.randint(0, 224 - h)
            left = np.random.randint(0, 224 - w)
            images[i, :, top:top+h, left:left+w] = 0.0

    return images


sample_images, sample_labels = next(iter(train_loader))

aug_weak = augment_batch(sample_images[:5], "weak")
aug_medium = augment_batch(sample_images[:5], "medium")
aug_strong = augment_batch(sample_images[:5], "strong")

print("weak:", aug_weak.shape)
print("medium:", aug_medium.shape)
print("strong:", aug_strong.shape)

## 13. 작은 데이터셋 전략 선택 가이드

오프라인에서도 개념은 동일하다.

```text
데이터가 매우 작음 → Full Freeze부터
데이터가 중간 → Partial Fine-tuning
데이터가 큼 → Full Fine-tuning 가능
도메인 차이 큼 → 뒤쪽 block부터 천천히 unfreeze
증강이 너무 강함 → 원본 의미가 깨질 수 있음
```

In [ ]:
decision_guide = {
    "very_small": "Full Freeze부터 시작한다",
    "medium": "Partial Fine-tuning을 기본 후보로 둔다",
    "large": "Full Fine-tuning도 가능하지만 regularization이 필요하다",
    "domain_gap_large": "Gradual Unfreezing으로 천천히 푼다",
    "augmentation_too_strong": "원본 의미가 깨질 수 있으므로 검증한다"
}

for key, value in decision_guide.items():
    print(f"{key}: {value}")

## 14. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `FakeData` | 가짜 이미지 데이터셋 | 인터넷 없이 구조 연습 |
| `weights=None` | pretrained weight 없이 모델 생성 | 다운로드 불필요 |
| `freeze` | 동결 | `requires_grad=False` |
| `partial` | 일부 해제 | layer4 + fc 학습 |
| `full` | 전체 학습 | 모든 parameter 학습 |
| `param_groups` | optimizer parameter group | group별 lr 지정 |
| `overfit gap` | train acc - test acc | 과적합 정도 |
| `augmentation` | 데이터 증강 | 입력 변형 |
| `RandomErasing` | 일부 영역 지우기 | strong 증강 |
| `Gradual Unfreezing` | 점진적 해제 | 뒤 layer부터 해제 |

## 15. 시험용 요약

```text
오프라인 버전 핵심 = 인터넷 없이도 transfer learning 전략의 코드 구조를 연습한다
```

꼭 기억할 것:

- 원본 강의는 CIFAR-10 다운로드와 pretrained ResNet18을 사용한다.
- 인터넷이 없으면 FakeData와 `weights=None`으로 구조를 연습할 수 있다.
- `weights=None`은 진짜 사전학습 효과가 아니다.
- Full Freeze는 fc만 학습한다.
- Partial Fine-tuning은 layer4와 fc를 학습한다.
- Full Fine-tuning은 전체 layer를 학습한다.
- 차등 학습률은 backbone에는 작은 lr, head에는 큰 lr을 준다.
- 작은 데이터셋은 Full Freeze 또는 Partial부터 시작하는 것이 안전하다.
- 증강은 강할수록 무조건 좋은 것이 아니다.